# Hypersonic Wing MDO

Wing skin sizing from Mach 0.8 to Mach 5, including thermal effects at hypersonic speeds — follows `demo_hypersonic_wing.py`.

- `WingGeometry` and `wing_panel_loads` for spanwise load distributions
- Adiabatic wall temperature and why it matters above Mach 3
- Thermal resultants and how they enter the Tsai-Wu check
- Buckling RF and when it overtakes the strength constraint
- Mass penalty breakdown across five Mach cases

**Five cases:** M0.8 subsonic, M1.7, M2.4, M5.0 mechanical only, M5.0 thermal+buckling.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from composite_panel import (
    IM7_8552, IM7_8552_thermal,
    WingGeometry, wing_panel_loads,
    aero_wall_temperature, equilibrium_wall_temperature,
    thermal_state_from_flight, thermal_resultants,
    Ply, Laminate,
    Nxx_cr, buckling_rf,
)
from composite_panel.optimizer import (
    optimize_wing, detect_balance_pairs,
    WingOptimizationResult,
)

---
## 1. Wing Geometry

`WingGeometry` defines a trapezoidal planform. The key parameters and what they control:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `semi_span` | 4.5 m | Distance from root to tip |
| `root_chord` | 4.0 m | Chord length at the root |
| `taper_ratio` | 0.25 | tip_chord / root_chord |
| `sweep_le_deg` | 50° | Leading-edge sweep (high sweep → hypersonic) |
| `t_over_c` | 0.04 | Thickness-to-chord ratio (thin section) |
| `mtow_n` | 120 kN | Max take-off weight (sets root bending moment) |

A 50° leading-edge sweep is aggressive — characteristic of a combat aircraft or hypersonic glide vehicle. It reduces wave drag at supersonic/hypersonic speeds but increases the chordwise component of aerodynamic loads.

In [ ]:
wing = WingGeometry(
    semi_span    = 4.5,
    root_chord   = 4.0,
    taper_ratio  = 0.25,
    sweep_le_deg = 50.0,
    t_over_c     = 0.04,
    mtow_n       = 120_000.0,
)

# Show how chord varies with span station eta = y/(b/2)
etas = np.linspace(0, 1, 11)
print('Spanwise chord distribution:')
print(f'  {"eta":>6}  {"chord [m]":>10}')
for e in etas:
    print(f'  {e:>6.2f}  {wing.chord(e):>10.3f}')

---
## 2. Material: Mechanical and Thermal Properties

At hypersonic speeds the skin heats up significantly. We need two material objects:

- `IM7_8552()` — standard elastic/strength properties for Tsai-Wu failure and stiffness calculations
- `IM7_8552_thermal()` — adds coefficients of thermal expansion (CTE) and cure temperature, needed to compute thermal running loads

The **cure temperature** is critical. IM7/8552 cures at ~177°C. After cure, the panel is stress-free at that temperature. In service, if the wall temperature exceeds the cure temperature, the cured-in compressive residual stresses become tensile — potentially reducing the effective RF.

In [ ]:
mat = IM7_8552()          # elastic + strength
pt  = IM7_8552_thermal()  # adds CTE, cure temp

print('Elastic properties:', mat)
print()
print(f'Cure temperature: {pt.T_cure - 273.15:.0f} C')
print(f'CTE fibre direction (alpha1): {pt.alpha1*1e6:.2f} ppm/K')
print(f'CTE transverse (alpha2):      {pt.alpha2*1e6:.2f} ppm/K')

---
## 3. Adiabatic Wall Temperature — Why Mach 5 is Different

At high speeds, aerodynamic heating raises the skin temperature. Two related temperatures:

**Adiabatic wall temperature** (`T_aw`) — the equilibrium temperature of an insulated (zero-heat-flux) surface. It depends only on Mach and altitude:

$$T_{aw} = T_{\infty} \left(1 + r \cdot \frac{\gamma - 1}{2} M^2\right)$$

where `r ≈ 0.85` is the turbulent recovery factor.

**Equilibrium wall temperature** (`T_eq`) — the temperature of a real (heat-conducting, radiating) surface. It also depends on the wetted length `x` from the leading edge because laminar/turbulent transition changes the heat transfer coefficient.

IM7/8552 has a practical service limit around 120-150°C. At Mach 5 / 20 km, `T_aw` typically exceeds 600°C — far above the matrix glass transition temperature. In practice you would use a ceramic matrix composite or a thermal protection system, but this demo shows what the mechanical sizing looks like when you include the thermal stress terms.

In [ ]:
ALT_M = 20_000   # flight altitude [m]

machs = [0.8, 1.7, 2.4, 5.0]
print(f'Adiabatic wall temperatures at {ALT_M/1e3:.0f} km:')
print(f'  {"Mach":>6}  {"T_aw [C]":>10}  {"T_aw [K]":>10}')
for m in machs:
    T = aero_wall_temperature(m, ALT_M)
    print(f'  {m:>6.1f}  {T-273.15:>10.1f}  {T:>10.1f}')

print(f'\nIM7/8552 cure temp: {pt.T_cure - 273.15:.0f} C')
print(f'  -> T_aw at M5 EXCEEDS cure temp by {aero_wall_temperature(5.0,ALT_M)-pt.T_cure:.0f} K')
print(f'     Thermal residuals flip sign: cured-in compression becomes tension')

---
## 4. How `thermal_resultants` Works

When the laminate temperature changes by `delta_T` from cure temperature, each ply tries to expand/contract according to its CTE. Because adjacent plies are bonded and have different orientations, they constrain each other — generating **thermal running loads** `N_T`.

$$N_T = \sum_k \bar{Q}^{(k)} \cdot \alpha^{(k)} \cdot \Delta T \cdot t_k$$

These `N_T` loads are added to the aerodynamic running loads before the Tsai-Wu check. At Mach 5, `delta_T` can be several hundred Kelvin, producing `N_T` values comparable to the mechanical loads.

In [ ]:
# Example: compute thermal resultants at the mid-span station, Mach 5
eta_mid  = 0.45
x_c_mid  = 0.4 * wing.chord(eta_mid)   # 40% chord location from leading edge
ts_mid   = thermal_state_from_flight(5.0, ALT_M, x_c_mid)

print(f'Thermal state at mid-span (eta={eta_mid}, x={x_c_mid:.2f}m):')
print(f'  T_wall  = {ts_mid.T_wall - 273.15:.1f} C')
print(f'  T_cure  = {pt.T_cure - 273.15:.1f} C')
print(f'  delta_T = {ts_mid.T_wall - pt.T_cure:.1f} K')

# Build a representative laminate to evaluate thermal resultants
ANGLES   = [0.0, 45.0, -45.0, 90.0]
ang_full = ANGLES + list(reversed(ANGLES))
n_ply    = len(ang_full)
t_demo   = 0.125e-3   # nominal ply thickness for illustration
plies_ex = [Ply(mat, t_demo, a) for a in ang_full]
lam_ex   = Laminate(plies_ex)

NT, _ = thermal_resultants(lam_ex.plies, [pt]*n_ply, ts_mid, lam_ex.z_interfaces)
print(f'\nThermal running loads N_T [kN/m]:')
print(f'  NT_xx = {NT[0]/1e3:+.1f} kN/m')
print(f'  NT_yy = {NT[1]/1e3:+.1f} kN/m')
print(f'  NT_xy = {NT[2]/1e3:+.1f} kN/m')

---
## 5. Buckling Reserve Factor

Thin panels under compression can buckle before the material reaches its strength limit. The critical buckling load for a simply-supported orthotropic plate is:

$$N_{xx}^{cr} = \frac{\pi^2}{b^2} \left[ D_{11} \left(\frac{b}{a}\right)^2 m^2 + 2(D_{12} + 2D_{66}) + D_{22} \left(\frac{a}{b}\right)^2 \frac{1}{m^2} \right]$$

where `m` is the number of half-waves (minimised over m=1,2,...), `a` is the panel length (span), `b` is the panel width (chord), and `D` is the bending stiffness matrix.

The **buckling RF** is simply `N_cr / N_applied`. A thin, lightly-loaded panel may pass Tsai-Wu with RF > 1.5 but still buckle (buckling RF < 1). Adding the buckling constraint forces the optimizer to use a thicker laminate (larger D-matrix), which can significantly increase mass.

In [ ]:
PANEL_A = 0.50   # rib pitch [m]
PANEL_B = 0.20   # stringer pitch [m]

# Show how buckling RF varies with laminate thickness for a fixed load
N_demo = np.array([-150e3, -30e3, 10e3])   # representative panel load [N/m]
n_plies_range = [4, 6, 8, 10, 12, 16]

print(f'Buckling RF vs laminate thickness (N={N_demo[0]/1e3:.0f} kN/m Nxx):')
print(f'  {"n_plies":>8}  {"h [mm]":>8}  {"RF_buckle":>10}')
for np_ in n_plies_range:
    angs = ([0, 45, -45, 90] * (np_ // 4))[:np_]
    pl   = [Ply(mat, t_demo, a) for a in angs]
    lm   = Laminate(pl)
    rf_b = buckling_rf(N_demo, lm.D, PANEL_A, PANEL_B)
    print(f'  {np_:>8d}  {lm.thickness*1e3:>8.3f}  {rf_b:>10.3f}')

---
## 6. Full Wing Optimisation — All Five Cases

`optimize_wing` runs the laminate optimiser at `N_STATIONS` span stations for a given Mach/altitude, collecting loads, running the NLP, and assembling total skin mass.

The optimizer variables are the ply thicknesses `t_half` (one per unique angle in the half-stack). Constraints:
- Tsai-Wu RF >= `rf_min` for every ply at every station
- Buckling RF >= `buckle_rf_min` when enabled
- Balance: `t(+theta) == t(-theta)` (enforced via `balance_pairs`)

In [ ]:
ANGLES     = [0.0, 45.0, -45.0, 90.0]
PAIRS      = detect_balance_pairs(ANGLES)
ETAS       = np.linspace(0.05, 0.95, 12)
N_STATIONS = len(ETAS)
ALPHA_DEG  = 3.0
N_LOAD     = 2.5

cases = {
    'M0.8  subsonic':     dict(mach=0.8, use_thermal=False, use_buckling=False),
    'M1.7  supersonic':   dict(mach=1.7, use_thermal=False, use_buckling=False),
    'M2.4  supersonic':   dict(mach=2.4, use_thermal=False, use_buckling=False),
    'M5.0  mech only':    dict(mach=5.0, use_thermal=False, use_buckling=False),
    'M5.0  therm+buckle': dict(mach=5.0, use_thermal=True,  use_buckling=True),
}

results_all = {}

for label, cfg in cases.items():
    mach = cfg['mach']
    print(f'  Sizing: {label} ...')

    ts_list = []
    for eta in ETAS:
        if cfg['use_thermal']:
            x_c = 0.4 * wing.chord(eta)
            ts_list.append(thermal_state_from_flight(mach, ALT_M, x_c))
        else:
            ts_list.append(None)

    result = optimize_wing(
        wing            = wing,
        mach            = mach,
        altitude_m      = ALT_M,
        alpha_deg       = ALPHA_DEG,
        mat             = mat,
        angles_half_deg = ANGLES,
        n_load          = N_LOAD,
        n_stations      = N_STATIONS,
        rf_min          = 1.5,
        t_min           = 0.05e-3,
        t_init          = 0.15e-3,
        balance_pairs   = PAIRS,
        rho_kg_m3       = 1600.0,
        panel_a         = PANEL_A if cfg['use_buckling'] else None,
        panel_b         = PANEL_B if cfg['use_buckling'] else None,
        buckle_rf_min   = 1.0,
        thermal_states  = ts_list if cfg['use_thermal'] else None,
        ply_thermal     = pt if cfg['use_thermal'] else None,
    )
    results_all[label] = result

print('\nDone.')

---
## 7. Mass Summary

How much does each flight regime cost in skin mass relative to the M0.8 baseline?

In [ ]:
m_base = results_all['M0.8  subsonic'].total_skin_mass
print(f'{"Case":<32}  {"Mass [kg]":>10}  {"vs M0.8":>8}')
print('─' * 58)
for lbl in cases:
    m   = results_all[lbl].total_skin_mass
    pct = (m - m_base) / m_base * 100
    sign = '+' if pct >= 0 else ''
    print(f'{lbl:<32}  {m:>10.1f}  {sign}{pct:>7.0f}%')

---
## 8. Visualisation — Spanwise Results

In [ ]:
BG    = '#0a0e1a'; WHITE = '#e8edf5'; DIM = '#3a4060'
colors_map = {
    'M0.8  subsonic':     '#4488ff',
    'M1.7  supersonic':   '#00ddbb',
    'M2.4  supersonic':   '#ff8833',
    'M5.0  mech only':    '#ff4455',
    'M5.0  therm+buckle': '#cc44ff',
}

def _style(ax, legend=False):
    ax.set_facecolor(BG)
    ax.tick_params(colors=WHITE, labelsize=8)
    ax.xaxis.label.set_color(WHITE); ax.yaxis.label.set_color(WHITE)
    ax.title.set_color(WHITE)
    for sp in ax.spines.values(): sp.set_edgecolor(DIM)
    ax.grid(color=DIM, linewidth=0.4, alpha=0.5)
    if legend:
        ax.legend(fontsize=7, framealpha=0.15, labelcolor=WHITE)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), facecolor=BG)
fig.suptitle(f'Wing Skin MDO  |  IM7/8552  [0/+/-45/90]s  |  M0.8->M5  |  Alt={ALT_M/1e3:.0f}km',
             color=WHITE, fontsize=10)
fig.subplots_adjust(wspace=0.35, top=0.83, bottom=0.14)

# Spanwise thickness
ax = axes[0]
for label in cases:
    wr = results_all[label]
    ax.plot(ETAS, wr.thicknesses * 1e3, color=colors_map[label], lw=2,
            marker='o', ms=3, label=label)
ax.set_xlabel('eta = y/b  [-]'); ax.set_ylabel('Laminate h [mm]')
ax.set_title('Spanwise thickness')
_style(ax, legend=True)

# Areal density penalty vs M0.8
ax = axes[1]
rho_base = results_all['M0.8  subsonic'].areal_densities
for label in cases:
    wr  = results_all[label]
    pct = (wr.areal_densities - rho_base) / rho_base * 100
    ax.plot(ETAS, pct, color=colors_map[label], lw=2, label=label)
ax.axhline(0, color=colors_map['M0.8  subsonic'], lw=1, ls='--', alpha=0.5)
ax.set_xlabel('eta = y/b  [-]'); ax.set_ylabel('Mass penalty vs M0.8 [%]')
ax.set_title('Areal density penalty')
_style(ax, legend=True)

# Total skin mass bar
ax = axes[2]
lbls = list(cases.keys())
masses = [results_all[l].total_skin_mass for l in lbls]
clrs   = [colors_map[l] for l in lbls]
bars = ax.barh(lbls, masses, color=clrs, edgecolor=DIM, alpha=0.85)
ax.bar_label(bars, fmt='%.1f kg', color=WHITE, fontsize=8, padding=4)
ax.set_xlabel('Semi-span upper-skin mass [kg]')
ax.set_title('Total skin mass')
_style(ax)

plt.show()

---
## 9. Wall Temperature vs Span (Mach 5)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), facecolor=BG)

T_eq = np.array([
    equilibrium_wall_temperature(5.0, ALT_M, x_station=0.4*wing.chord(e))
    for e in ETAS
])

for M_ref, color, lbl in [(0.8, '#4488ff', 'T_aw M0.8'), (1.7, '#00ddbb', 'T_aw M1.7'),
                           (2.4, '#ff8833', 'T_aw M2.4'), (5.0, '#ff4455', 'T_aw M5.0')]:
    T = aero_wall_temperature(M_ref, ALT_M)
    ax.axhline(T - 273.15, color=color, lw=1.2, ls='--', label=lbl)

ax.plot(ETAS, T_eq - 273.15, color='#cc44ff', lw=2, label='T_wall equil. M5')
ax.axhline(pt.T_cure - 273.15, color=WHITE, lw=1.0, ls=':',
           label=f'T_cure {pt.T_cure-273:.0f}C')
ax.fill_between(ETAS, pt.T_cure - 273.15, T_eq - 273.15, alpha=0.10, color='#cc44ff')

ax.set_xlabel('eta = y/b  [-]', color=WHITE)
ax.set_ylabel('Temperature [C]', color=WHITE)
ax.set_title('Adiabatic wall temperature vs span', color=WHITE)
ax.set_facecolor(BG)
ax.tick_params(colors=WHITE)
for sp in ax.spines.values(): sp.set_edgecolor(DIM)
ax.grid(color=DIM, lw=0.4, alpha=0.5)
ax.legend(fontsize=8, framealpha=0.15, labelcolor=WHITE)
plt.tight_layout()
plt.show()

print(f'\nShaded region = temperature range where T > T_cure')
print(f'In this region, cured-in residual stresses change sign.')

---
## 10. Key Takeaways

1. **Mach number drives skin mass non-linearly.** Moving from M0.8 to M2.4 grows mass primarily because dynamic pressure increases (more Nyy). Moving from M2.4 to M5 adds thermal loads on top of even higher dynamic pressure.

2. **Thermal loads at Mach 5 can rival or exceed mechanical loads.** The equilibrium wall temperature at M5 / 20 km exceeds the IM7/8552 cure temperature, so the cured-in compressive residuals flip to tensile. This reduces the effective margin against transverse (sigma2) tension failure.

3. **Buckling is the second-order constraint.** The `M5.0 therm+buckle` case adds more mass than `M5.0 mech only` because the panel must be thick enough for adequate bending stiffness (D-matrix), not just strength. This effect is most pronounced near the root where the panel is widest.

4. **Spanwise thickness is not constant.** Bending moments are largest at the root — so root stations drive the highest Nxx and need the most material. Tip stations see lower loads and converge towards minimum gauge.

5. **The optimizer respects balance pairs.** Forcing `t(+45) == t(-45)` is not optional — without balance, the A16 and A26 terms are non-zero, causing coupled extension-shear response that is undesirable in production laminates.